In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math

def create_euclidean_graph(
    n: int,
    density: str,
    undirected_density: float = 0.50,
    penalty_pct: float = 0.75,
    seed: int | None = None,
) -> nx.DiGraph:

    rng = np.random.default_rng(seed)
    coords = rng.random((n, 2))

    UG_full = nx.Graph()
    for idx, (x, y) in enumerate(coords):
        UG_full.add_node(idx, pos=(x, y))
    for i in range(n):
        for j in range(i + 1, n):
            d = float(np.linalg.norm(coords[i] - coords[j]))
            UG_full.add_edge(i, j, weight=round(d, 4))

    if density == "dense":
        UG = UG_full
    elif density == "sparse":
        # start from the MST so the graph stays connected, then add random edges
        UG = nx.minimum_spanning_tree(UG_full, weight="weight")
        max_e = n * (n - 1) // 2
        target_e = int(np.ceil(undirected_density * max_e))
        candidates = list(set(UG_full.edges()) - set(UG.edges()))
        rng.shuffle(candidates)
        need = max(0, target_e - UG.number_of_edges())
        for (u, v) in candidates[:need]:
            UG.add_edge(u, v, weight=UG_full[u][v]["weight"])
    else:
        raise ValueError("density must be 'sparse' or 'dense'")

    assign_priority_weights(UG)

    DG = nx.DiGraph()
    for u, data in UG.nodes(data=True):
        DG.add_node(u, **data)
    for u, v, d in UG.edges(data=True):
        w = d["weight"]
        p = d["priority"]
        pen = round(penalty_pct * w, 4)
        DG.add_edge(u, v, weight=w, penalty=pen, priority=p)
        DG.add_edge(v, u, weight=w, penalty=pen, priority=p)

    return DG

def assign_priority_weights(G: nx.DiGraph) -> None:
    """Priority = edge betweenness scaled to integers 1-10."""
    bet = nx.edge_betweenness_centrality(G, normalized=True, weight="weight")
    vals = np.array(list(bet.values()), dtype=float)
    range_ = np.ptp(vals)
    if range_ == 0:
        scaled = np.zeros_like(vals)
    else:
        scaled = (vals - vals.min()) / range_

    ints = (1 + 9 * scaled).round().astype(int)

    for (u, v), pr in zip(bet, ints):
        G[u][v]["priority"] = int(pr)

def show_graph(G: nx.DiGraph, title: str = "") -> None:
    pos = {n: G.nodes[n]["pos"] for n in G}
    plt.figure(figsize=(4, 4))
    nx.draw_networkx_nodes(G, pos, node_size=250, node_color="#1f78b4")
    nx.draw_networkx_edges(G, pos, arrows=True, width=0.8)
    nx.draw_networkx_labels(G, pos, font_size=8, font_color="white")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import dimod

def build_qubo(G: nx.DiGraph, K: int, budget: int, P1: float, P2: float):
    arcs, nodes = list(G.edges()), list(G.nodes())

    C_max     = max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in arcs)
    bits_dual = math.ceil(math.log2(C_max + 1))
    bits_s    = math.ceil(math.log2(budget + 1))

    var_index, idx = {}, 0

    # ------- x_{ij}
    for arc in arcs:
        var_index[('x', arc)] = idx; idx += 1

    # ------- pi_j (j != 0)
    for j in nodes:
        if j == 0: continue
        for l in range(bits_dual):
            var_index[('pi', j, l)] = idx; idx += 1

    # ------- lambda_nk (n != 0)
    for n in nodes:
        if n == 0: continue
        for k in range(K):
            for l in range(bits_dual):
                var_index[('lam', n, k, l)] = idx; idx += 1

    # ------- mu_k, nu_k
    for k in range(K):
        for l in range(bits_dual):
            var_index[('mu', k, l)] = idx; idx += 1
            var_index[('nu', k, l)] = idx; idx += 1

    # ------- two sided slack r_plus / r_minus
    for (i, j) in arcs:
        for k in range(K):
            for l in range(bits_dual):
                var_index[('r_plus',  i, j, k, l)] = idx; idx += 1
                var_index[('r_minus', i, j, k, l)] = idx; idx += 1

    # ------- global budget slack s
    for l in range(bits_s):
        var_index[('s_global', l)] = idx; idx += 1

    # ------- build QUBO
    Q = {i: {} for i in range(idx)}
    def add(u, v, c):
        if u > v: u, v = v, u
        Q[u][v] = Q[u].get(v, 0.0) + c

    # dual reward
    for j in nodes:
        if j == 0: continue
        for l in range(bits_dual):
            u = var_index[('pi', j, l)]
            add(u, u, -2**l)
    for k in range(K):
        for l in range(bits_dual):
            add(var_index[('mu', k, l)], var_index[('mu', k, l)], -2**l)
            add(var_index[('nu', k, l)], var_index[('nu', k, l)], -2**l)

    # flow penalty
    for (i, j) in arcs:
        c, d = G[i][j]['weight'], G[i][j]['penalty']
        xi   = var_index[('x', (i, j))]
        for k in range(K):
            terms = [(xi, d)]
            if j != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('pi', j, l)], -2**l))
            if j != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('lam', j, k, l)], -2**l))
            if i != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('lam', i, k, l)],  2**l))
            if i == 0:
                for l in range(bits_dual):
                    terms.append((var_index[('mu', k, l)], -2**l))
            if j == 0:
                for l in range(bits_dual):
                    terms.append((var_index[('nu', k, l)], -2**l))
            for l in range(bits_dual):
                terms.append((var_index[('r_plus',  i, j, k, l)], -2**l))
                terms.append((var_index[('r_minus', i, j, k, l)],  2**l))

            for (u, w) in terms:
                add(u, u, P1 * (w*w + 2*c*w))
            for a in range(len(terms)):
                u, wu = terms[a]
                for v, wv in terms[a+1:]:
                    add(u, v, 2*P1*wu*wv)

    # budget penalty
    s_bits = [var_index[('s_global', l)] for l in range(bits_s)]
    terms  = [(var_index[('x', (i, j))], 1) for (i, j) in arcs] + \
             [(bit, 2**l) for l, bit in enumerate(s_bits)]
    for u, w in terms:
        add(u, u, P2 * (w*w - 2*budget*w))
    for a in range(len(terms)):
        u, wu = terms[a]
        for v, wv in terms[a+1:]:
            add(u, v, 2*P2*wu*wv)

    bqm = dimod.BinaryQuadraticModel.from_qubo(
        {(u, v): c for u, row in Q.items() for v, c in row.items()})
    return bqm, var_index

In [ ]:
def improved_qubo(bqm_base: dimod.BinaryQuadraticModel,
                  var_index: dict,
                  G: nx.DiGraph,
                  alpha: float = 5.0
                  ) -> dimod.BinaryQuadraticModel:
    bqm = bqm_base.copy()
    for (u, v) in G.edges():
        idx = var_index[('x', (u, v))]
        # 1) direct interdiction reward: subtract the true penalty d_{ij}
        d = G[u][v]['penalty']
        bqm.add_variable(idx, -d)
        # 2) priority reward: subtract alpha * p_ij
        p = G[u][v]['priority']
        bqm.add_variable(idx, -alpha * p)
    return bqm

In [ ]:
def choose_penalties(G: nx.DiGraph, K: int, safety_factor: float = 10.0):
    """
    Pick P1 and P2 so any single constraint breach costs more than
    flipping every interdiction bit could ever gain in the original QUBO.
    """
    arcs  = list(G.edges())

    C_max = max(G[u][v]["weight"] + G[u][v]["penalty"] for u,v in arcs)

    worst_orig_gain = len(arcs) * C_max

    P1 = math.ceil(safety_factor * worst_orig_gain * K)

    P2 = math.ceil(safety_factor * worst_orig_gain * len(arcs))

    return P1, P2

In [ ]:
def choose_scale_minbits(G, min_bits=5, max_bits=9):
    """
    Smallest power-of-two scale factor that gives min_bits <= bits_dual <= max_bits.
    """
    max_val = max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in G.edges())
    for exp in range(max_bits + 1):
        factor = 2 ** exp
        bits   = math.ceil(math.log2(max_val * factor + 1))
        if bits >= min_bits and bits <= max_bits:
            return factor, bits
    return 2 ** max_bits, max_bits

In [ ]:
def check_feasibility(sample, var_index, G, K, budget, verbose=True):
    inv = {idx: key for key, idx in var_index.items()}

    x_sum = sum(sample[idx] for idx, k in inv.items() if k[0] == 'x')
    s_val = sum(sample[idx]*2**inv[idx][-1] for idx in sample
                if inv[idx][0] == 's_global')
    budget_ok = (x_sum + s_val == budget)

    bits_dual = math.ceil(math.log2(
        max(G[u][v]['weight']+G[u][v]['penalty'] for u, v in G.edges()) + 1))

    flow_ok, first_bad = True, None
    for (i, j) in G.edges():
        for k in range(K):
            pi = sum(sample.get(var_index.get(('pi', j, l), -1), 0)*2**l
                     for l in range(bits_dual)) if j != 0 else 0
            lam_j = sum(sample.get(var_index.get(('lam', j, k, l), -1), 0)*2**l
                        for l in range(bits_dual)) if j != 0 else 0
            lam_i = sum(sample.get(var_index.get(('lam', i, k, l), -1), 0)*2**l
                        for l in range(bits_dual)) if i != 0 else 0
            mu = sum(sample.get(var_index.get(('mu', k, l), -1), 0)*2**l
                     for l in range(bits_dual)) if i == 0 else 0
            nu = sum(sample.get(var_index.get(('nu', k, l), -1), 0)*2**l
                     for l in range(bits_dual)) if j == 0 else 0
            r_plus  = sum(sample.get(var_index.get(('r_plus',  i, j, k, l), -1),0)*2**l
                          for l in range(bits_dual))
            r_minus = sum(sample.get(var_index.get(('r_minus', i, j, k, l), -1),0)*2**l
                          for l in range(bits_dual))
            lhs = (G[i][j]['weight']
                   + G[i][j]['penalty']*sample[var_index[('x', (i, j))]]
                   - pi - lam_j + lam_i - mu - nu
                   - r_plus + r_minus)
            if lhs != 0:
                flow_ok, first_bad = False, ((i, j), k, lhs)
                break
        if not flow_ok: break

    if verbose:
        print("\n---- Feasibility report ----")
        print(f"budget OK : {budget_ok}")
        print(f"flow   OK : {flow_ok}")
        if not flow_ok:
            arc, veh, res = first_bad
            print(f"  first flow imbalance on arc {arc}, veh {veh}, residual={res}")
        print("-----------------------------\n")
    return budget_ok and flow_ok

In [ ]:
def repair_to_feasible(sample: dict[int, int],
                       var_index: dict,
                       G: nx.DiGraph,
                       K: int) -> None:
    """
    Flip slack bits until every flow residual is zero.
    Handles the case where both r_plus & r_minus of a weight are already 1
    by turning the opposite bit off.
    Mutates sample in place.
    """
    bits_dual = math.ceil(math.log2(
        max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in G.edges()) + 1))

    while True:
        changed = False

        for (i, j) in G.edges():
            for k in range(K):

                c, d = G[i][j]['weight'], G[i][j]['penalty']
                x    = sample[var_index[('x', (i, j))]]

                def bit_sum(prefix, cond=True):
                    return sum(
                        sample.get(var_index.get(prefix+(l,), -1), 0) * (1 << l)
                        for l in range(bits_dual)
                    ) if cond else 0

                pi     = bit_sum(('pi',  j), j != 0)
                lam_j  = bit_sum(('lam', j, k), j != 0)
                lam_i  = bit_sum(('lam', i, k), i != 0)
                mu     = bit_sum(('mu',  k), i == 0)
                nu     = bit_sum(('nu',  k), j == 0)
                r_plus = bit_sum(('r_plus',  i, j, k))
                r_minus= bit_sum(('r_minus', i, j, k))

                gap = c + d*x - pi - lam_j + lam_i - mu - nu - r_plus + r_minus
                if gap == 0:
                    continue

                delta = abs(gap)
                for l in range(bits_dual):
                    if not (delta & (1 << l)):
                        continue

                    w        = 1 << l
                    key_plus = ('r_plus',  i, j, k, l)
                    key_minus= ('r_minus', i, j, k, l)
                    idx_plus = var_index[key_plus]
                    idx_minus= var_index[key_minus]

                    if gap > 0:
                        if sample[idx_plus] == 0:
                            sample[idx_plus] = 1
                        elif sample[idx_minus] == 0:
                            sample[idx_minus] = 1
                        else:
                            sample[idx_minus] = 0
                        gap -= w
                        changed = True

                    else:
                        if sample[idx_minus] == 0:
                            sample[idx_minus] = 1
                        elif sample[idx_plus] == 0:
                            sample[idx_plus] = 1
                        else:
                            sample[idx_plus] = 0
                        gap += w
                        changed = True

        if not changed:
            break

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# QiskitRuntimeService.save_account(channel="ibm_cloud", overwrite=True, token="")

In [ ]:
import pandas as pd
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.translators import to_ising
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz
from scipy.optimize import minimize
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp

import time

def dimod_bqm_to_qp(bqm):
    labels = list(bqm.variables)
    name_of  = {lbl: f"v{lbl}" for lbl in labels}
    label_of = {name: lbl for lbl, name in name_of.items()}

    qp = QuadraticProgram()
    for lbl in labels:
        qp.binary_var(name=name_of[lbl])

    linear = {name_of[lbl]: float(bias) for lbl, bias in bqm.linear.items()}

    quad = {}
    for (u, v), w in bqm.quadratic.items():
        a, b = name_of[u], name_of[v]
        if a > b:
            a, b = b, a
        quad[(a, b)] = quad.get((a, b), 0.0) + float(w)

    qp.minimize(linear=linear, quadratic=quad)
    return qp, name_of, label_of

def _qaoa_solve_qp(
    qp,
    reps=1,
    shots=2000,
    seed=123,
    use_runtime=True,
    backend=None,
):
    print(f"[QAOA] Starting solve: vars={qp.get_num_vars()}, reps={reps}, shots={shots}, runtime={use_runtime}")

    H, _ = to_ising(qp)
    ansatz = QAOAAnsatz(cost_operator=H, reps=reps)

    circ_opt = ansatz.decompose().decompose()
    n_qubits = circ_opt.num_qubits
    local_qubit_cap = 24

    if n_qubits <= local_qubit_cap:
        sim_local = AerSimulator()
        local_shots = 128

        def _mean_energy_from_probs(H: SparsePauliOp, probs: dict[str, float]) -> float:
            n = H.num_qubits
            ev = 0.0
            z_cache = {}
            for bitstr, p in probs.items():
                b = bitstr[::-1]
                z = np.array([1 - 2*int(b[i]) for i in range(n)], dtype=float)
                z_cache[bitstr] = (z, p)
            for pauli, coeff in zip(H.paulis, H.coeffs):
                label = pauli.to_label()
                idx = [i for i, ch in enumerate(label[::-1]) if ch == 'Z']
                if not idx:
                    term_ev = 1.0
                else:
                    term_ev = 0.0
                    for _, (z, p) in z_cache.items():
                        term_ev += p * float(np.prod(z[idx]))
                ev += float(coeff.real) * term_ev
            return ev

        rng    = np.random.default_rng(seed)
        theta0 = rng.uniform(0, 2*np.pi, len(circ_opt.parameters))

        def energy(theta):
            circ_bound = circ_opt.assign_parameters(theta)
            circ_meas  = circ_bound.copy()
            circ_meas.measure_all()
            tq = transpile(circ_meas, backend=sim_local, optimization_level=0)
            result = sim_local.run(tq, shots=local_shots, seed_simulator=seed).result()
            counts = result.get_counts()
            total  = sum(counts.values())
            probs  = {k: v/total for k, v in counts.items()} if total > 0 else {}
            if not probs:
                raise RuntimeError("No samples from AerSimulator.")
            return _mean_energy_from_probs(H, probs)

        print("[QAOA] Optimizing angles locally...")
        opt_res   = minimize(energy, theta0, method="COBYLA", options={"maxiter": 60, "rhobeg": 0.2})
        theta_opt = opt_res.x
        print("[QAOA] Optimization complete")
    else:
        print(f"[QAOA] Skipping local optimization: {n_qubits} qubits > local cap {local_qubit_cap}")
        rng    = np.random.default_rng(seed)
        theta_opt = rng.uniform(0, 2*np.pi, len(circ_opt.parameters))
        print("[QAOA] Using random initial angles for hardware sampling")

    if use_runtime:
        service = QiskitRuntimeService()
        be = backend or service.least_busy(simulator=False)
        print(f"[QAOA] Using backend: {be.name}, qubits={be.num_qubits}")
        if ansatz.num_qubits > be.num_qubits:
            print(f"[Guard] circuit needs {ansatz.num_qubits} qubits but {be.name} has {be.num_qubits}")
            return None

        circ_hw = transpile(
            ansatz,
            target=be.target,
            initial_layout=list(range(ansatz.num_qubits)),
            layout_method="trivial",
            routing_method="sabre",
            optimization_level=1
        )

        print(f"[QAOA] Transpiled circuit width: {circ_hw.num_qubits}")
        print("[QAOA] Running sampler on hardware...")
        sampler = RuntimeSampler(mode=be)
        circ_sample_hw = circ_hw.assign_parameters(theta_opt)
        circ_sample_hw = circ_sample_hw.copy()
        circ_sample_hw.measure_all()
        data = sampler.run([circ_sample_hw], shots=shots).result()[0].data
        print("[QAOA] Sampling complete")
    else:
        print("[QAOA] Running sampler on simulator...")
        sampler = AerSampler()
        circ_sample_sim = circ_opt.assign_parameters(theta_opt)
        circ_sample_sim = circ_sample_sim.copy()
        circ_sample_sim.measure_all()
        data = sampler.run([circ_sample_sim], shots=shots).result()[0].data
        print("[QAOA] Sampling complete")

    counts = data.meas.get_counts()
    if not counts:
        raise RuntimeError("Sampler returned no counts")
    best_bitstring = max(counts, key=counts.get)

    names  = [v.name for v in qp.variables]
    bits_le = list(best_bitstring[::-1])
    print("[QAOA] Solve complete")
    return {names[i]: int(bits_le[i]) for i in range(len(names))}

def solve_bqm_on_ibm(bqm, reps=1, shots=2000, seed=123, use_runtime=True, backend=None, verbose=True):
    t0 = time.time()
    qp, name_of, label_of = dimod_bqm_to_qp(bqm)
    n_vars = qp.get_num_vars()

    if use_runtime:
        service = QiskitRuntimeService()
        be = backend or service.least_busy(simulator=False)
        if verbose:
            print(f"[solve] Backend={be.name} qubits={be.num_qubits} | problem vars={n_vars}")
        if n_vars > be.num_qubits:
            print(f"[Guard] {n_vars} vars > {be.name} qubits ({be.num_qubits}). Skipping (hardware-only).")
            return None
        sol_by_name = _qaoa_solve_qp(qp, reps=reps, shots=shots, seed=seed,
                                     use_runtime=True, backend=be)
    else:
        if verbose:
            print(f"[solve] Using simulator | problem vars={n_vars}")
        sol_by_name = _qaoa_solve_qp(qp, reps=reps, shots=shots, seed=seed,
                                     use_runtime=False, backend=None)

    res = {label_of[name]: int(val) for name, val in sol_by_name.items()}
    if verbose:
        print(f"[solve] Finished in {time.time()-t0:.2f}s\n")
    return res

In [ ]:
def defender_cost(arcs, G):
    H = G.copy()
    for u, v in arcs:
        H[u][v]['weight'] += H[u][v]['penalty']
    return max(nx.single_source_dijkstra_path_length(H, 0, weight='weight').values())

def evaluate_metrics(G, K, B, alpha):
    print(f"\n[eval] nodes={G.number_of_nodes()} edges={G.number_of_edges()} K={K} B={B} alpha={alpha:.3f}")
    t0 = time.time()

    SCALE, bits_dual = choose_scale_minbits(G, min_bits=7, max_bits=9)
    for u, v in G.edges():
        G[u][v]['weight']  = int(round(G[u][v]['weight']  * SCALE))
        G[u][v]['penalty'] = int(round(G[u][v]['penalty'] * SCALE))
    P1, P2 = choose_penalties(G, K, safety_factor=5)

    bqm_base, vidx = build_qubo(G, K, B, P1, P2)
    bqm_imp  = improved_qubo(bqm_base, vidx, G, alpha)
    print(f"[eval] BQMs built: base_vars={bqm_base.num_variables}, imp_vars={bqm_imp.num_variables} (t+{time.time()-t0:.2f}s)")

    t1 = time.time()
    mb = solve_bqm_on_ibm(bqm_base, use_runtime=True, reps=3, shots=2000, verbose=True)
    if mb is None:
        print("[eval] Base instance too large -> skip")
        return np.nan, np.nan, np.nan
    print(f"[eval] base solution obtained (t+{time.time()-t1:.2f}s)")

    t2 = time.time()
    repair_to_feasible(mb, vidx, G, K)
    E_base = bqm_base.energy(mb)
    print(f"[eval] base repaired + energy={E_base:.3f} (t+{time.time()-t2:.2f}s)")

    t3 = time.time()
    mi = solve_bqm_on_ibm(bqm_imp, use_runtime=True, reps=3, shots=2000, verbose=True)
    if mi is None:
        print("[eval] Improved instance too large -> skip")
        return np.nan, np.nan, np.nan
    print(f"[eval] improved solution obtained (t+{time.time()-t3:.2f}s)")

    t4 = time.time()
    repair_to_feasible(mi, vidx, G, K)
    E_imp_on_base = bqm_base.energy(mi)
    print(f"[eval] improved repaired + energy_on_base={E_imp_on_base:.3f} (t+{time.time()-t4:.2f}s)")

    delta_q = 100 * (E_base - E_imp_on_base) / abs(E_base)
    arcs_base = [(u, v) for u, v in G.edges() if mb[vidx[('x', (u, v))]]]
    arcs_imp  = [(u, v) for u, v in G.edges() if mi[vidx[('x', (u, v))]]]
    c_base = defender_cost(arcs_base, G)
    c_imp  = defender_cost(arcs_imp, G)
    print(f"[eval] delta_q={delta_q:.2f}% base_cost={c_base:.3f} imp_cost={c_imp:.3f} (total t={time.time()-t0:.2f}s)")

    return delta_q, c_base, c_imp

def quality_vs_graph_size(node_list, K, B, reps=1):
    defender_trials = []
    print("\n---- Quality vs Graph Size ----")
    for n in node_list:
        for t in range(reps):
            G = create_euclidean_graph(n, density="sparse", seed=t)
            costs = np.array([G[u][v]['weight'] for u, v in G.edges()])
            prios = np.array([G[u][v]['priority'] for u, v in G.edges()])
            alpha = costs.mean() / prios.mean()
            _, c_base, c_imp = evaluate_metrics(G, K, B, alpha)
            defender_trials.append({"nodes": n, "trial": t, "model": "Base",     "cost": c_base})
            defender_trials.append({"nodes": n, "trial": t, "model": "Improved", "cost": c_imp})
            print(f"Nodes: {n} Cycle: {t} completed")
        print()
    return pd.DataFrame(defender_trials)

def quality_vs_vehicle_amount(node_size, fleet_list, B, reps=1):
    defender_trials = []
    print("\n---- Quality vs Vehicle Amount ----")
    for K in fleet_list:
        for t in range(reps):
            G = create_euclidean_graph(node_size, density="sparse", seed=t)
            costs = np.array([G[u][v]['weight'] for u, v in G.edges()])
            prios = np.array([G[u][v]['priority'] for u, v in G.edges()])
            alpha = costs.mean() / prios.mean()
            _, c_base, c_imp = evaluate_metrics(G, K, B, alpha)
            defender_trials.append({"fleet": K, "trial": t, "model": "Base",     "cost": c_base})
            defender_trials.append({"fleet": K, "trial": t, "model": "Improved", "cost": c_imp})
            print(f"Vehicles: {K} Cycle: {t} completed")
        print()
    return pd.DataFrame(defender_trials)

In [ ]:
import matplotlib.ticker as mtick

def average_nodes_and_fleets(df_defender_nodes: pd.DataFrame,
                             df_defender_fleet: pd.DataFrame) -> pd.DataFrame:
    a = df_defender_nodes[['trial','model','cost']].copy()
    b = df_defender_fleet[['trial','model','cost']].copy()

    a['trial'] = pd.to_numeric(a['trial'], errors='coerce')
    b['trial'] = pd.to_numeric(b['trial'], errors='coerce')
    a['cost']  = pd.to_numeric(a['cost'], errors='coerce')
    b['cost']  = pd.to_numeric(b['cost'], errors='coerce')

    a = a.dropna().groupby(['trial','model'], as_index=False)['cost'].mean()\
         .rename(columns={'cost':'cost_nodes'})
    b = b.dropna().groupby(['trial','model'], as_index=False)['cost'].mean()\
         .rename(columns={'cost':'cost_fleet'})

    merged = pd.merge(a, b, on=['trial','model'], how='inner')
    merged['cost'] = (merged['cost_nodes'] + merged['cost_fleet']) / 2.0

    df_avg = merged[['trial','model','cost']].groupby(['trial','model'], as_index=False)['cost'].mean()
    return df_avg.sort_values(['trial','model'])

def plot_averaged_defender(df_avg, title="Defender Cost vs Trial (n=3, K=1)"):
    df = df_avg.copy()
    df['trial'] = pd.to_numeric(df['trial'], errors='coerce')
    df['cost']  = pd.to_numeric(df['cost'], errors='coerce')
    df = df.dropna(subset=['trial','cost'])

    plt.figure(figsize=(7,4.5))

    for model, color in [("Base", "tab:blue"), ("Improved", "tab:orange")]:
        sub = df[df['model'] == model].sort_values('trial')
        plt.plot(
            sub['trial'].to_numpy(),
            sub['cost'].to_numpy(),
            '-o',
            linewidth=2,
            markersize=6,
            color=color,
            label=model
        )

    plt.title(title)
    plt.xlabel("Trial")
    plt.ylabel("Defender Cost")
    plt.gca().xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
    plt.legend()
    plt.tight_layout()
    plt.show()